# 🎯 [MOV02_M02] 도로 파괴, 교량 붕괴, 통제 구역 등 전장 제약 조건 그래프 전처리 모듈 구현
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M02 |
| **미션 주제** | 도로 파괴, 교량 붕괴, 통제 구역 등 전장 제약 조건 그래프 전처리 모듈 구현 |
| **소요 시간** | 약 2.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 도로 파괴, 교량 붕괴, 통제 구역 등 전장 제약 조건 그래프 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

도로 파괴, 통제 구역 등 전장 제약 조건 그래프 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **도로 파괴, 교량 붕괴·하중 제한, 통제 구역 보고를 여러 형식에서 읽어 도로망 그래프에 반영하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **CSV, 엑셀 내보내기(탭 구분), JSON, 공문 텍스트 형식의 가상 보고서를 각각 파싱해 표준 제약 레코드로 바꾼다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/mov02_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""도로 파괴·교량 붕괴·하중 제한·통제 구역 보고를 여러 형식(CSV·엑셀 내보내기·JSON·공문 텍스트)의 가상 문서에서 파싱해 도로망 그래프에 반영하는 전장 제약 조건 전처리 모듈을 구현하고, 파싱 성공률과 반영 후 연결성을 확인한다."""
import heapq
import csv
import io
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
E = {e["id"]: e for e in edges}
DEPOT, UNITS = "N0000", [f"N{r:02d}07" for r in range(6)]

# 1) [교육용가상자료] 메모리 안의 가상 보고 문서 4종 (파일 대신 문자열로 모의)
CSV_DOC = """edge_id,status,reason
E017,파괴,포격으로 노면 유실
E040,파괴,지뢰 의심 신고
E058,정상,복구 완료
E071,파괴,차량 잔해 적치"""
EXCEL_DOC = "구간ID\t교량하중(t)\t비고\nE025\t15\t교각 손상\nE064\t20\t임시 가설 교량"      # 엑셀 내보내기(탭 구분, 한글 머리글)
JSON_DOC = json.dumps({"control_zones": [{"zone": "CZ-1", "x_km": 17.5, "y_km": 12.0, "r_km": 2.5, "reason": "화생방 오염 의심"},
                                         {"zone": "CZ-2", "x_km": 30.0, "y_km": 21.0, "r_km": 1.8, "reason": "민간인 대피 통로"}]}, ensure_ascii=False)
MEMO_DOC = """[가상 공문] 제목: 도로 상황 변경 통보
1. 교량 E046 붕괴로 통행 불가.
2. E083 구간 교량 하중 제한 20톤으로 하향.
3. E999 구간 파괴 보고(구간 번호 확인 필요)."""

ALIAS = {"edge_id": ["edge_id", "구간ID", "구간"], "limit_t": ["limit_t", "교량하중(t)", "하중"], "status": ["status", "상태"]}


def std_cols(df):
    """양식이 바뀌어도(머리글 변경) 별칭 사전으로 표준 컬럼명에 맞춘다."""
    ren = {c: k for k, al in ALIAS.items() for c in df.columns if c.strip() in al}
    return df.rename(columns=ren)


def parse(fmt, text):
    """형식별 파서. 결과는 표준 제약 레코드 목록 {type, edge_id | zone, value, reason}."""
    if fmt == "csv":
        df = std_cols(pd.DataFrame(list(csv.DictReader(io.StringIO(text)))))
        return [{"type": "close" if s == "파괴" else "reopen", "edge_id": e, "reason": r} for e, s, r in zip(df.edge_id, df.status, df.reason)]
    if fmt == "excel":
        df = std_cols(pd.DataFrame(list(csv.DictReader(io.StringIO(text), delimiter="\t"))))
        return [{"type": "limit", "edge_id": e, "value": float(v), "reason": r} for e, v, r in zip(df.edge_id, df.limit_t, df["비고"])]
    if fmt == "json":
        return [{"type": "zone", "zone": z["zone"], "value": (z["x_km"], z["y_km"], z["r_km"]), "reason": z["reason"]}
                for z in json.loads(text)["control_zones"]]
    if fmt == "memo":
        out = []
        for line in text.splitlines():
            m = re.search(r"(E\d{3}).*?(붕괴|파괴|통행 불가)", line)
            if m:
                out.append({"type": "close", "edge_id": m.group(1), "reason": line.strip()}); continue
            m = re.search(r"(E\d{3}).*?하중 제한 (\d+)\s*톤", line)
            if m:
                out.append({"type": "limit", "edge_id": m.group(1), "value": float(m.group(2)), "reason": line.strip()})
        return out
    raise ValueError(f"지원하지 않는 형식: {fmt}")


def check_rec(r):
    """스키마 유효성 검사: 구간ID 형식·존재 여부, 하중 1~100t, 통제 구역 반경 0.1~10km."""
    if r["type"] == "zone":
        return [] if 0.1 <= r["value"][2] <= 10 else [f"{r['zone']} 반경 범위 오류"]
    errs = [] if re.match(r"^E\d{3}$", r["edge_id"]) else [f"{r['edge_id']} 형식 오류"]
    if r["edge_id"] not in E:
        errs.append(f"{r['edge_id']} 존재하지 않는 구간")
    if r["type"] == "limit" and not 1 <= r["value"] <= 100:
        errs.append(f"{r['edge_id']} 하중 범위 오류")
    return errs


def apply(edges, recs, log):
    """제약 레코드를 그래프에 반영. 없는 구간 ID는 반영하지 않고 로그에 남긴다."""
    new = [dict(e) for e in edges]
    idx = {e["id"]: e for e in new}
    for r in recs:
        if r["type"] == "zone":
            x, y, rad = r["value"]
            for e in new:
                if edge_risk(e, nodes, [{"x": x, "y": y, "r": rad, "p": 1.0}]) > 0:
                    e["open"] = False; e["zone"] = r["zone"]
            log.append(f"{r['zone']} 통제 구역 반영")
        elif r["edge_id"] not in idx:
            log.append(f"{r['edge_id']} 없는 구간 ID - 반영 보류(확인 요청)")
        elif r["type"] == "close":
            idx[r["edge_id"]]["open"] = False
        elif r["type"] == "reopen":
            idx[r["edge_id"]]["open"] = True
        elif r["type"] == "limit":
            idx[r["edge_id"]].update(bridge=True, limit_t=r["value"])
    return new


docs = [("csv", CSV_DOC), ("excel", EXCEL_DOC), ("json", JSON_DOC), ("memo", MEMO_DOC)]
recs, rows = [], []
for fmt, text in docs:
    try:
        r = parse(fmt, text); recs += r; rows.append({"형식": fmt, "파싱": "성공", "레코드": len(r)})
    except Exception as ex:
        rows.append({"형식": fmt, "파싱": f"실패({ex})", "레코드": 0})
res = pd.DataFrame(rows)
rate = (res["파싱"] == "성공").mean()
print(res.to_string(index=False))
print(f"파싱 성공률 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'}), 제약 레코드 {len(recs)}건")

log = []
after = apply(edges, recs, log)
print("\n[반영 로그]", " / ".join(log))
print(f"통행 불가 구간 {sum(not e['open'] for e in after)}개, 하중 제한 교량 {sum(e['bridge'] for e in after)}개 (원본 {sum(e['bridge'] for e in edges)}개)")

# 2) 반영 전후 연결성·도달성 점검 (트럭 총중량 15t, 트레일러 총중량 50t 가정)
ids = list(nodes)
pos = {n: i for i, n in enumerate(ids)}


def n_components(es, gross):
    ok = [e for e in es if passable(e, gross)]
    g = coo_matrix((np.ones(len(ok)), ([pos[e["u"]] for e in ok], [pos[e["v"]] for e in ok])), shape=(len(ids), len(ids)))
    return connected_components(g, directed=False)[0]


print("\n차종      반영 전 연결 요소  반영 후 연결 요소  도달 가능 부대(후)")
for kind, gross in [("truck", 15), ("trailer", 50)]:
    adj = build_adj(nodes, after, lambda e: edge_minutes(e, kind) if passable(e, gross) else None)
    reach = sum(dijkstra(adj, DEPOT, u)[0] < float("inf") for u in UNITS)
    print(f"{kind:8s}  {n_components(edges, gross):16d}  {n_components(after, gross):16d}  {reach}/{len(UNITS)}")

# 3) 양식 변경 대응 시험: 머리글이 바뀐 CSV도 별칭 사전으로 처리되는지 확인
changed = "구간,상태,reason\nE017,파괴,머리글 변경 양식"
print("\n머리글 변경 양식 파싱:", parse("csv", changed))
errs = [x for r in recs for x in check_rec(r)]
print(f"스키마 검사: 레코드 {len(recs)}건 중 오류 {len(errs)}건 {errs} -> 해당 건은 반영 보류 후 확인 요청")

# 4) 반영 전후 그래프 그림
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, es, title in [(ax[0], edges, "Road network (before)"), (ax[1], after, "After battlefield constraints")]:
    for e in es:
        (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
        col = "red" if not e["open"] else ("orange" if e["bridge"] else ("tab:blue" if e["road"] == "paved" else "tab:gray"))
        a.plot([x1, x2], [y1, y2], color=col, lw=2.5 if not e["open"] else 1.5, ls="--" if not e["open"] else "-")
    a.plot(*nodes[DEPOT], "ks", ms=10)
    for u in UNITS:
        a.plot(*nodes[u], "g^", ms=9)
    a.set_title(title); a.set_xlabel("x (km)"); a.set_ylabel("y (km)"); a.set_aspect("equal")
for r in recs:
    if r["type"] == "zone":
        ax[1].add_patch(plt.Circle(r["value"][:2], r["value"][2], color="purple", alpha=0.2))
ax[1].text(0, -4, "red dashed = closed, orange = bridge (weight limit), purple = control zone", fontsize=8)
plt.suptitle("MOV02 constraint preprocessing (synthetic network)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
   형식 파싱  레코드
  csv 성공    4
excel 성공    2
 json 성공    2
 memo 성공    3
파싱 성공률 100% (목표 100%: 달성), 제약 레코드 11건

[반영 로그] CZ-1 통제 구역 반영 / CZ-2 통제 구역 반영 / E999 없는 구간 ID - 반영 보류(확인 요청)
통행 불가 구간 10개, 하중 제한 교량 17개 (원본 14개)

차종      반영 전 연결 요소  반영 후 연결 요소  도달 가능 부대(후)
truck                    1                 2  6/6
trailer                  1                 2  6/6

머리글 변경 양식 파싱: [{'type': 'close', 'edge_id': 'E017', 'reason': '머리글 변경 양식'}]
스키마 검사: 레코드 11건 중 오류 1건 ['E999 존재하지 않는 구간'] -> 해당 건은 반영 보류 후 확인 요청
```
</details>
